# core

> Kosha is a tool for building a context for code generation based on your repo and environment.
> It uses a vector database to store code snippets and their embeddings, allowing you to search for relevant code based
> on natural language queries.
> The core functionality includes managing the database, updating package metadata, embedding code snippets,
> and performing context-aware searches.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import ast, os, re
from ast import get_source_segment as gs
from collections import defaultdict, Counter
from functools import lru_cache
from apswutils.utils import hash_record
from importlib.metadata import version as v, metadata as meta, distribution as dist
from chonkie import AutoEmbeddings, BaseEmbeddings
from fastcore.all import (Path, first, patch, timed_cache, L, merge, AttrDict, bind, num_cpus,
                          filter_keys, not_, in_, listify, true, fdelegates, type2str, parallel_async)
from fastcore.xdg import xdg_data_home
from tqdm import tqdm
from json import loads as jl
from litesearch import *

## Utils
> Utility functions for finding the repo root, copying the SKILL.md file, running async code from sync contexts,
> and getting installed package versions.

In [ ]:
#| export
@timed_cache(maxsize=4096)
def parse(code=None, p=None):
    "Parse source, tag parents, return (tree, imp, top_fns, all_fns, imp2mod). Cached — called freely."
    if not code and not p: return None, {}, set(), set(), {}
    try: tree = ast.parse(Path(p).read_text(errors='replace') if not code else code)
    except SyntaxError: return None, {}, set(), set(), {}
    [setattr(c,'parent',n) for n in ast.walk(tree) for c in ast.iter_child_nodes(n)]
    is_top = lambda n: isinstance(getattr(n,'parent',None), ast.Module)
    is_fn  = lambda n: isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef))
    is_nm  = lambda n: isinstance(n, ast.Name)
    imp, imp2mod, top = {}, {}, set()
    # Single pass: imports + top-level fn names
    for n in ast.walk(tree):
        if is_fn(n) and is_top(n): top.add(n.name)
        elif isinstance(n, ast.Import):
            for a in n.names: imp[a.asname or a.name.split('.')[0]] = a.name.split('.')[0]
        elif isinstance(n, ast.ImportFrom):
            pkg = (n.module or '').split('.')[0]
            for a in n.names:
                if a.name != '*':
	                imp[a.asname or a.name] = pkg
	                if n.module: imp2mod[a.asname or a.name] = f'{n.module}.{a.name}'
                else: imp.setdefault(f'*{pkg}', pkg)
    ca  = {nm for n in tree.body if isinstance(n,ast.Assign) and isinstance(n.value,ast.Call)
           for t in n.targets
           for nm in ([t.id] if is_nm(t) else [e.id for e in t.elts if is_nm(e)] if isinstance(t,ast.Tuple) else [])}
    return tree, imp, top, top | ca, imp2mod

In [ ]:
#| export
def repo_root() -> Path:
	'Find the root of the current git repository, or None if not in a repo.'
	return first((Path.cwd(), *Path.cwd().parents), lambda p: (p/'.git').exists())

def mv_skill_md(dry_run=True, dir=None) -> None:
	'Copy bundled SKILL.md to skill directories.'
	base = Path(__file__).parent if '__file__' in globals() else Path.cwd()
	if not (src := base.joinpath('SKILL.md')).exists(): return
	root = dir or repo_root() or '.'
	ts = [Path(root)/'.agents/skills/kosha/SKILL.md', Path('.claude/skills/kosha/SKILL.md')]
	if dry_run: print(f'Would copy {src} to: {list(map(str,ts))}')
	else:
		[p.mk_write(src.read_text(encoding='utf-8'))for p in ts]
		print(f'Installed → {list(map(str,ts))}')

In [ ]:
#| export
def arun(coro) -> any:
	'Run an async coroutine from sync code, even if already inside an event loop'
	import asyncio
	try: asyncio.get_running_loop()
	except RuntimeError: return L(asyncio.run(coro))
	import concurrent.futures
	with concurrent.futures.ThreadPoolExecutor() as pool: return L(pool.submit(asyncio.run, coro).result())

In [ ]:
async def _add(a, b): return a + b
assert arun(_add(1, 2)) == L(3)
print('arun: sync context ok')

arun: sync context ok


In [ ]:
async def _test_nested():
	result = arun(_add(10, 20))
	assert first(result) == 30, f'got {result}'
arun(_test_nested())
print('arun: nested-loop context ok')

arun: nested-loop context ok


In [ ]:
mv_skill_md()

Would copy /Users/71293/code/personal/orgs/kosha/nbs/SKILL.md to: ['/Users/71293/code/personal/orgs/kosha/.agents/skills/kosha/SKILL.md', '.claude/skills/kosha/SKILL.md']


In [ ]:
mv_skill_md(dir='.')

Would copy /Users/71293/code/personal/orgs/kosha/nbs/SKILL.md to: ['.agents/skills/kosha/SKILL.md', '.claude/skills/kosha/SKILL.md']


In [ ]:
#| export
_req_nm = re.compile(r'^[\w][\w.-]*')
_spec_cached = lru_cache(maxsize=None)(spec)

def pkg_trans_deps(seeds:list, depth:int=2) -> L:
	'BFS over importlib.metadata requires: return seeds + all transitive deps up to depth levels (installed only, no optional extras).'
	seen, frontier = set(seeds), set(seeds)
	for _ in range(depth):
		nxt = set()
		for pkg in frontier:
			try:
				for req in (dist(pkg).requires or []):
					if 'extra ==' in req: continue
					if (m := _req_nm.match(req)) and _spec_cached(m.group(0)): nxt.add(m.group(0))
			except Exception: pass
		nxt -= seen
		if not nxt: break
		seen |= nxt; frontier = nxt
	return L(seen)

In [ ]:
#| export
def env_pkg_versions(pyproject=True, depth:int=1) -> dict:
	'''Get a dict of installed package versions using importlib.metadata.
	passing depth traverse multiple layers of dependencies'''
	pkgs = installed_packages(pyproject=pyproject)
	if pyproject and depth: pkgs = pkg_trans_deps(pkgs, depth)
	return {dist(p).metadata['Name']: dist(p).version for p in pkgs}

In [ ]:
installed_packages(pyproject=True)

['fastprogress', 'litesearch', 'pyan3', 'pyskills', 'watchfiles']

In [ ]:
pkg_trans_deps(['litesearch'], depth=1)

['pdf-oxide', 'pandas', 'chonkie', 'notebook', 'pillow', 'usearch', 'codesigs', 'model2vec', 'onnxruntime', 'tokenizers', 'yake', 'litesearch', 'fastlite']

In [ ]:
# httpx depends on httpcore; depth=1 should include both httpx + httpcore
d1 = pkg_trans_deps(['httpx'], depth=1)
assert 'httpx' in d1, "seeds must be in result"
assert 'httpcore' in d1, f"depth=1 should include httpcore (direct dep of httpx): {d1}"
# depth=0 returns only seeds
assert list(pkg_trans_deps(['httpx'], depth=0)) == ['httpx'], "depth=0 should return only seeds"
# pyproject seeds → depth=2 expansion
pyp = installed_packages(pyproject=True)
if pyp:
    deep = pkg_trans_deps(list(pyp), depth=2)
    assert set(pyp).issubset(set(deep)), "seeds must be a subset of result"
    assert len(deep) >= len(pyp), "depth=2 should expand beyond pyproject direct deps"
    print(f"pkg_trans_deps ok: {len(pyp)} pyproject seeds → {len(deep)} packages at depth=2")
else:
    print("pkg_trans_deps ok (no pyproject deps to test)")

pkg_trans_deps ok: 5 pyproject seeds → 60 packages at depth=2


In [ ]:
env_pkg_versions()

{'fastprogress': '1.1.6',
 'fastcore': '2.0.1',
 'fastaudit': '0.2.5',
 'anyio': '4.14.1',
 'codesigs': '0.0.3',
 'Jinja2': '3.1.6',
 'yake': '0.7.3',
 'litesearch': '0.0.30',
 'chonkie': '1.7.0',
 'watchfiles': '1.2.0',
 'notebook': '7.6.0',
 'python-fasthtml': '0.14.5',
 'pdf_oxide': '0.3.73',
 'pyskills': '0.0.19',
 'pandas': '3.0.3',
 'pillow': '12.3.0',
 'usearch': '2.25.3',
 'model2vec': '0.8.2',
 'onnxruntime': '1.27.0',
 'tokenizers': '0.23.1',
 'fastlite': '0.2.4',
 'pyan3': '2.6.1'}

## embedder
> jina-embeddings-v2-base-code is a model designed for code understanding and generation tasks

In [ ]:
#| export
bge_instr = AttrDict(document="{text}",query="{text}")
bge_model = AttrDict(model='TaylorAI/bge-micro-v2', onnx_path='onnx/model_quantized.onnx',prompt=bge_instr, tti=True)
strict_skip_folder_re = r'^[._]|^(?:tests?|examples?|docs?|build|dist)$'
strict_skip_file_re = r'^[._]|^(?:setup\.py|conftest\.py)$'

@timed_cache(3600*24)
@fdelegates(FastEncode)
def embedder(
    emb_model=bge_model,  # model config AttrDict (model name, onnx_path, prompt instructions)
	**kwargs
) -> 'FastEncode':
	'''Load or return cached CodeRankEmbed ONNX document embedder (24h TTL).
	Hardware-aware: tunes parallel/batch_size to host (override via KOSHA_EMBED_PARALLEL / KOSHA_EMBED_BATCH).'''
	import psutil
	par = int(os.environ.get('KOSHA_EMBED_PARALLEL') or min(4,(num_cpus() or 1)))
	bs  = int(os.environ.get('KOSHA_EMBED_BATCH') or 64)
	if ram_gb := psutil.virtual_memory().available/(1024**3): bs = min(bs, int(ram_gb//0.15))
	try: return FastEncode(emb_model, parallel=par, batch_size=bs, **kwargs)
	except Exception: return FastEncode(emb_model, parallel=1, batch_size=bs, **kwargs)

@timed_cache(3600*24)
def static_embedder(
    emb_model='minishlab/potion-code-16M-v2',  # HuggingFace model id for static embeddings
	**kwargs
) -> 'BaseEmbeddings':
    'Load or return cached static (potion-retrieval) embedder (24h TTL).'
    return AutoEmbeddings().get_embeddings(emb_model)

def emb_doc(
    e  # embedder factory — callable returning a FastEncode or AutoEmbeddings instance
):
    'Curry an embedder factory into a document-embedding function `t,**kw -> embedding`.'
    return lambda t, **kw: e().encode_document(t,**kw) if isinstance(e(), FastEncode) else e().embed_batch(listify(t))

def emb_query(
    e  # embedder factory — callable returning a FastEncode or AutoEmbeddings instance
):
    'Curry an embedder factory into a query-embedding function `t,**kw -> embedding`.'
    return lambda t, **kw: e().encode_query(t,**kw) if isinstance(e(), FastEncode) else e().embed_batch(listify(t))

In [ ]:
fe = embedder()

In [ ]:
emb_doc(embedder)('def add(a, b): return a + b')

array([[-0.1525   , -0.02603  ,  0.009476 , -0.0452   , -0.063    ,
         0.01869  , -0.05945  ,  0.04126  ,  0.04858  ,  0.01467  ,
         0.0165   , -0.0404   ,  0.03058  ,  0.012695 ,  0.05176  ,
        -0.01229  , -0.05298  ,  0.05573  , -0.0752   , -0.03564  ,
        -0.00155  ,  0.00819  ,  0.004192 , -0.01636  ,  0.02516  ,
         0.05765  , -0.05746  , -0.0376   , -0.006638 , -0.2008   ,
         0.00895  ,  0.04468  ,  0.05905  , -0.02405  , -0.04227  ,
        -0.0505   , -0.04514  ,  0.05444  , -0.05615  ,  0.03375  ,
         0.03845  , -0.008965 ,  0.01544  , -0.03152  , -0.06107  ,
        -0.04935  , -0.0384   , -0.001802 ,  0.010216 , -0.03717  ,
         0.01888  , -0.03714  , -0.01213  , -0.014626 ,  0.02234  ,
         0.04868  ,  0.02145  ,  0.04892  , -0.003267 ,  0.04456  ,
         0.0635   ,  0.03134  , -0.0708   ,  0.0817   ,  0.02145  ,
         0.02718  , -0.02718  , -0.01976  ,  0.011375 ,  0.08923  ,
        -0.0776   ,  0.02118  ,  0.04636  ,  0.0

In [ ]:
be = bind(embedder,emb_model=bge_model)

In [ ]:
emb_doc(be)('def add(a, b): return a + b')

array([[-0.1525   , -0.02603  ,  0.009476 , -0.0452   , -0.063    ,
         0.01869  , -0.05945  ,  0.04126  ,  0.04858  ,  0.01467  ,
         0.0165   , -0.0404   ,  0.03058  ,  0.012695 ,  0.05176  ,
        -0.01229  , -0.05298  ,  0.05573  , -0.0752   , -0.03564  ,
        -0.00155  ,  0.00819  ,  0.004192 , -0.01636  ,  0.02516  ,
         0.05765  , -0.05746  , -0.0376   , -0.006638 , -0.2008   ,
         0.00895  ,  0.04468  ,  0.05905  , -0.02405  , -0.04227  ,
        -0.0505   , -0.04514  ,  0.05444  , -0.05615  ,  0.03375  ,
         0.03845  , -0.008965 ,  0.01544  , -0.03152  , -0.06107  ,
        -0.04935  , -0.0384   , -0.001802 ,  0.010216 , -0.03717  ,
         0.01888  , -0.03714  , -0.01213  , -0.014626 ,  0.02234  ,
         0.04868  ,  0.02145  ,  0.04892  , -0.003267 ,  0.04456  ,
         0.0635   ,  0.03134  , -0.0708   ,  0.0817   ,  0.02145  ,
         0.02718  , -0.02718  , -0.01976  ,  0.011375 ,  0.08923  ,
        -0.0776   ,  0.02118  ,  0.04636  ,  0.0

In [ ]:
emb_doc(static_embedder)('def add(a, b): return a + b')

array([[ 1.0394e-01,  4.9438e-03, -1.2079e-01,  6.4026e-02,  6.0699e-02,
        -1.9971e-01,  7.8125e-02, -2.9053e-02,  1.3100e-02,  3.9398e-02,
         1.1307e-02, -8.7830e-02,  1.8845e-02,  4.8309e-02, -8.6670e-02,
         8.5083e-02, -1.0788e-02,  4.5197e-02, -1.8661e-02,  4.2084e-02,
         2.5284e-02, -1.0582e-02, -2.4757e-03, -2.8519e-02,  3.7109e-02,
        -2.4109e-02, -5.1514e-02, -3.3875e-02, -3.2867e-02,  1.3110e-01,
        -7.6342e-04, -5.1331e-02, -2.2522e-02, -8.9233e-02, -1.1131e-02,
        -1.0872e-02, -5.7861e-02, -8.6975e-02, -3.4790e-02, -1.3191e-02,
         7.6965e-02, -3.4210e-02, -3.2532e-02,  3.8605e-02,  4.3304e-02,
        -9.4360e-02, -4.0497e-02, -5.2490e-02,  2.0142e-02, -1.2463e-01,
        -1.7975e-02,  4.4464e-02,  5.1788e-02,  1.8530e-01,  9.9976e-02,
         7.2571e-02,  7.6233e-02,  1.0156e-01,  6.5727e-03,  1.8646e-02,
         3.8109e-03,  6.3232e-02,  4.9774e-02,  4.4373e-02,  8.8196e-03,
         1.0364e-01,  2.7695e-02, -1.1450e-01,  4.9

## Kosha
> The main class for managing the code and environment databases, updating package metadata, embedding code snippets,
> and performing context-aware searches.

In [ ]:
#| export
class Kosha:
	'Kosha allows you to build a context for code generation based on your repo and environment.'
	def __init__(self, dir: Path = None, install_skill: bool = False, xdg_dir: Path = None, efn=static_embedder):
		self.root, self.xdg = Path(dir or repo_root() or '.'), xdg_dir or xdg_data_home()
		self.efn = bind(efn, md=self.xdg/f'kosha/model')
		self.emb_doc, self.emb_query = emb_doc(self.efn), emb_query(self.efn)
		if install_skill: mv_skill_md(dir=self.root, dry_run=False)
		self.cp, self.ce = self.root.joinpath('.kosha','code.db'), self.xdg.joinpath('kosha','env.db')
		for p in (self.cp, self.ce): p.parent.mkdir(parents=True, exist_ok=True)
		self.codedb, self.envdb = database(self.cp), database(self.ce)
		self.code_st,self.env_st = self.codedb.get_store(path=str,hash=True),self.envdb.get_store(package=str,hash=True)
		self.pkg_st, self.pkgs = self.envdb.get_store('pkg_store', hash=True), self.envdb.t.packages
		self.env_pd, self.code_rd = self.envdb.t.pkg_deps, self.codedb.t.repo_deps
		self.pkgs.create(name=str, version=str, summary=str, uploaded_at=float, pk=['name','version'],
			 defaults=dict(uploaded_at='CURRENT_TIMESTAMP'), if_not_exists=True)
		self.env_pd.create(from_pkg=str, to_pkg=str, n_modules=int, if_not_exists=True, pk=['from_pkg','to_pkg'])
		self.code_rd.create(from_module=str, to_pkg=str, n_files=int, if_not_exists=True, pk=['from_module','to_pkg'])

In [ ]:
k1=Kosha(xdg_dir=repo_root()/'.kosha',efn=embedder)

In [ ]:
k2=Kosha(xdg_dir=repo_root()/'.statickosha')

In [ ]:
k=Kosha()

In [ ]:
k.emb_doc('def add(a, b): return a + b')[:5]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 97165.34it/s]


array([[ 1.0394e-01,  4.9438e-03, -1.2079e-01,  6.4026e-02,  6.0699e-02,
        -1.9971e-01,  7.8125e-02, -2.9053e-02,  1.3100e-02,  3.9398e-02,
         1.1307e-02, -8.7830e-02,  1.8845e-02,  4.8309e-02, -8.6670e-02,
         8.5083e-02, -1.0788e-02,  4.5197e-02, -1.8661e-02,  4.2084e-02,
         2.5284e-02, -1.0582e-02, -2.4757e-03, -2.8519e-02,  3.7109e-02,
        -2.4109e-02, -5.1514e-02, -3.3875e-02, -3.2867e-02,  1.3110e-01,
        -7.6342e-04, -5.1331e-02, -2.2522e-02, -8.9233e-02, -1.1131e-02,
        -1.0872e-02, -5.7861e-02, -8.6975e-02, -3.4790e-02, -1.3191e-02,
         7.6965e-02, -3.4210e-02, -3.2532e-02,  3.8605e-02,  4.3304e-02,
        -9.4360e-02, -4.0497e-02, -5.2490e-02,  2.0142e-02, -1.2463e-01,
        -1.7975e-02,  4.4464e-02,  5.1788e-02,  1.8530e-01,  9.9976e-02,
         7.2571e-02,  7.6233e-02,  1.0156e-01,  6.5727e-03,  1.8646e-02,
         3.8109e-03,  6.3232e-02,  4.9774e-02,  4.4373e-02,  8.8196e-03,
         1.0364e-01,  2.7695e-02, -1.1450e-01,  4.9

In [ ]:
k2.emb_doc('def add(a, b): return a + b')[:5]

array([[ 1.0394e-01,  4.9438e-03, -1.2079e-01,  6.4026e-02,  6.0699e-02,
        -1.9971e-01,  7.8125e-02, -2.9053e-02,  1.3100e-02,  3.9398e-02,
         1.1307e-02, -8.7830e-02,  1.8845e-02,  4.8309e-02, -8.6670e-02,
         8.5083e-02, -1.0788e-02,  4.5197e-02, -1.8661e-02,  4.2084e-02,
         2.5284e-02, -1.0582e-02, -2.4757e-03, -2.8519e-02,  3.7109e-02,
        -2.4109e-02, -5.1514e-02, -3.3875e-02, -3.2867e-02,  1.3110e-01,
        -7.6342e-04, -5.1331e-02, -2.2522e-02, -8.9233e-02, -1.1131e-02,
        -1.0872e-02, -5.7861e-02, -8.6975e-02, -3.4790e-02, -1.3191e-02,
         7.6965e-02, -3.4210e-02, -3.2532e-02,  3.8605e-02,  4.3304e-02,
        -9.4360e-02, -4.0497e-02, -5.2490e-02,  2.0142e-02, -1.2463e-01,
        -1.7975e-02,  4.4464e-02,  5.1788e-02,  1.8530e-01,  9.9976e-02,
         7.2571e-02,  7.6233e-02,  1.0156e-01,  6.5727e-03,  1.8646e-02,
         3.8109e-03,  6.3232e-02,  4.9774e-02,  4.4373e-02,  8.8196e-03,
         1.0364e-01,  2.7695e-02, -1.1450e-01,  4.9

In [ ]:
#| export
def pkg_url(pkg: str) -> str | None:
    'Return best web URL for pkg from importlib.metadata: Source Code > Repository > Home-page > first Project-URL.'
    m, urls = meta(pkg), {}
    for e in (m.get_all('Project-URL') or []):
        label, _, u = e.partition(',')
        urls[label.strip().lower()] = u.strip()
    return (urls.get('source code') or urls.get('repository') or urls.get('source') or
            urls.get('github') or m.get('Home-page') or next(iter(urls.values()), None))

In [ ]:
#| export
def pkg_doc(pkg) -> dict:
	'Build pkg_store content dict: content=summary+readme, metadata=json.'
	m = meta(pkg)
	_get, _req_re, fn = lambda k: m.get(k,''), re.compile(r'^[\w-]+'), lambda rm: rm.group(0).lower().replace('-','_')
	reqs = ' '.join({fn(rm) for r in (m.get_all('Requires-Dist') or []) if (rm:=_req_re.match(r))})
	summary = _get('Summary')
	cont = summary+'\n'+(m.get_payload() or _get('Description') or '')[:2000].strip()
	return dict(content=cont, metadata=dict(name=pkg, version=v(pkg), keywords=_get('Keywords'),
	                                        requires=reqs, summary=summary, url=pkg_url(pkg)))

In [ ]:
pkg_doc('httpx')

{'content': 'The next generation HTTP client.\n<p align="center">\n  <a href="https://www.python-httpx.org/"><img width="350" height="208" src="https://raw.githubusercontent.com/encode/httpx/master/docs/img/butterfly.png" alt=\'HTTPX\'></a>\n</p>\n\n<p align="center"><strong>HTTPX</strong> <em>- A next-generation HTTP client for Python.</em></p>\n\n<p align="center">\n<a href="https://github.com/encode/httpx/actions">\n    <img src="https://github.com/encode/httpx/workflows/Test%20Suite/badge.svg" alt="Test Suite">\n</a>\n<a href="https://pypi.org/project/httpx/">\n    <img src="https://badge.fury.io/py/httpx.svg" alt="Package version">\n</a>\n</p>\n\nHTTPX is a fully featured HTTP client library for Python 3. It includes **an integrated command line client**, has support for both **HTTP/1.1 and HTTP/2**, and provides both **sync and async APIs**.\n\n---\n\nInstall HTTPX using pip:\n\n```shell\n$ pip install httpx\n```\n\nNow, let\'s get started:\n\n```pycon\n>>> import httpx\n>>> r = 

In [ ]:
u = pkg_url('httpx')
assert u and u.startswith('http'), f'expected a URL, got {u!r}'
print('pkg_url httpx:', u)
# packages without Project-URL fall back to Home-page
assert pkg_url('fastcore') is not None, 'fastcore should have a URL'
print('pkg_url fastcore:', pkg_url('fastcore'))

pkg_url httpx: https://github.com/encode/httpx
pkg_url fastcore: https://github.com/AnswerDotAI/fastcore/


In [ ]:
#| export
def has_init(d: Path) -> bool:
	'True if dir is a Python package root: has __init__.py or a C-extension __init__*.so.'
	return (d / '__init__.py').exists() or bool(list(d.glob('__init__*.so')))

def imp_root(p: Path) -> Path:
	'Import root of p: the nearest ancestor directory without __init__.py.'
	r = Path(p).parent
	while has_init(r): r = r.parent
	return r


In [ ]:
#| export
def enrich_chunks(content: L) -> L:
	'Set public_api+docstring flags on non-assign chunks and explode ClassDefs into method sub-chunks.'
	_m, _c = lambda d, k: d.get('metadata',{}).get(k,''), lambda d: d.get('content','')
	assigns, non_assigns = content.partition(lambda d: d['metadata'].get('type','') in ('Assign','AnnAssign'))
	chk = lambda e: isinstance(e, ast.Constant) and isinstance(e.value, str)
	def _get__all(d):
		try: return set(L(ast.parse(_c(d)).body[0].value.elts).filter(chk).attrgot('value'))
		except: return set()
	all_map = merge(*assigns.filter(lambda d: '__all__' in _c(d)).map(lambda d: {_m(d,'path'): _get__all(d)}))
	fwa = set(all_map)
	def _enrich(d):
		nm, pth, src = _m(d,'name'), _m(d,'path'), _c(d)
		tree, *_ = parse(src)
		try: doc = ast.get_docstring(tree.body[0]) if tree and tree.body else None
		except: doc = None
		pub = dict(docstring=doc,public_api=(nm in all_map[pth]) if pth in fwa else bool(nm and not nm.startswith('_')))
		return d | {'metadata': d['metadata'] | pub}
	def _cls_methods(d):
		src, m = _c(d), d['metadata']
		tree, *_ = parse(src)
		if tree is None or not tree.body: return L()
		cls = tree.body[0]
		if not isinstance(cls, ast.ClassDef): return L()
		pub, mod, off = m.get('public_api',False), m.get('mod_name',''), (m.get('lineno') or 1)-1
		is_fn = lambda n: isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and gs(src, n)
		mk = lambda n: d | {'content': gs(src, n).strip(), 'metadata': m | dict(
			name=n.name, type=type2str(n.__class__), mod_name=f'{mod}.{n.name}',
			public_api=pub and (not n.name.startswith('_') or n.name.startswith('__')),
			docstring=ast.get_docstring(n), lineno=off+n.lineno, end_lineno=off+(n.end_lineno or n.lineno))}
		return L(cls.body).filter(is_fn).map(mk)
	enriched = non_assigns.map(_enrich)
	return enriched + enriched.filter(lambda d: d['metadata'].get('type')=='ClassDef').map(_cls_methods).concat()

In [ ]:
#| export
os.environ['TOKENIZERS_PARALLELISM']='false'  # to suppress warnings from tokenizers
@patch
def nuke(self:Kosha):
	'Reset the databases by dropping all tables.'
	for p in (self.cp, self.ce): p.parent.delete()

@patch
def _is_pkg_ingested(self:Kosha, pkg:str) -> bool:
	'Check if a package is already ingested and up-to-date.'
	return first(self.pkgs(select='name, version', where=f'name={pkg!r} and version={v(pkg)!r}'))

def embed_chunk(chunk:list|L,emb_fn=emb_doc(embedder), **kwargs):
	'Embed a list of code chunks using emb_fn, batched to bound peak memory.'
	if not chunk: return
	c = [b for b in chunk if b['content'] and b['content'].strip()]
	if not c: return
	embs = emb_fn(L(c).attrgot('content'), **kwargs)
	for e,b in zip(embs, c): b['embedding'] = e.tobytes()
	return c

@fdelegates(embed_chunk)
def process_content(store, content:L, embed:bool=True, **kwargs):
	'Embed chunks and upsert into store; no-op if content is empty.'
	if not content: return
	if embed: content = embed_chunk(content, **kwargs)
	return store.insert_all(content, upsert=True, hash_id='id', hash_id_columns=['content'])

@patch
@fdelegates(process_content)
def process_env(self:Kosha, content=None, reembed=False, **kwargs):
	'Embed all documents in the env store, or only those without embeddings if reembed=False.'
	content = content or self.env_st(where=f'embedding is NULL' if not reembed else None)
	return process_content(self.env_st, content, emb_fn=self.emb_doc, **kwargs)

def _slug(s): return hash_record({"content": s}) if s else None

def _content(store, where:str=None, f=None):
	'Helper to retrieve content from a store with optional filtering and transformation.'
	c=L(store(select='content', where=where)).filter(lambda c: c['content'] and c['content'].strip()).attrgot('content')
	return c if not f else L(c).map(f)

def count_imp(files, own:str='') -> Counter:
	'External top-level imports from a source-string iterable. Reuses cached _parse.'
	c, fn = Counter(), lambda p: p and p!=own and p[0]!='_'
	def do(f): c.update(set(L(parse(p=f)[1].values()).filter(fn)))
	L(files).map(do)
	return c

In [ ]:
#| export
@patch
@fdelegates(pkg2chunks)
def update_pkg(self:Kosha, pkg:str, embed=True, exts=code_exts, verbose=True, force=False, imports=False, **kwargs):
    'Update package metadata in the packages table.'
    assert (o:= spec(pkg)), f'pkg {pkg} is not in environment'
    if verbose: print(f'updating pkg: {pkg} ...')
    ep = self._is_pkg_ingested(pkg)
    if ep and len(self.env_st(where=f'package={pkg!r}')) > 0:
        if verbose: print(f'package {ep} already loaded.')
        if not force: return
        print(f'forcing an update to pkg {ep}')
    pkg_par = Path(o.origin).parent.parent if o.origin else Path(o.submodule_search_locations[0])
    mod_fn = lambda p, n: '.'.join(list(Path(p).relative_to(pkg_par).with_suffix('').parts) + ([n] if n else []))
    _get = lambda o,k1 = None: o.get('metadata').get(k1,'')
    meta_fn = lambda d: d['metadata'] | dict(mod_name=mod_fn(_get(d,'path'), _get(d,'name') or None), package=pkg, version=v(pkg))
    fn = lambda d: dict(package=pkg, uploaded_at=_get(d,'uploaded_at'), metadata=meta_fn(d))
    files = pkg2files(pkg, exts=exts, **kwargs)
    cnt = [file_parse(f, assigns=True, imports=imports) for f in tqdm(files,f'parse files from {pkg}')]
    if cnt:
	    cnt, doc = enrich_chunks(L(cnt).concat().filter(true).map(lambda d: d | fn(d))), pkg_doc(pkg)
	    ex = L() if force else _content(self.env_st, where=f'package={pkg!r}', f=_slug)
	    cont_hash = {_slug(d['content']): d for d in cnt}
	    if dids:=set(ex).difference(cont_hash.keys()): self.env_st.delete_where(where='id in ("%s")'%'","'.join(dids))
	    ch = filter_keys(cont_hash, not_(in_(ex))).values()
	    if ch and self.process_env(ch, embed=embed):
		    pyf = L(files).filter(lambda f: str(f).endswith('.py'))
		    counts = count_imp(pyf, pkg.replace('-','_').split('.')[0])
		    rows = [dict(from_pkg=pkg, to_pkg=dep, n_modules=n) for dep,n in counts.items() if _spec_cached(dep)]
		    if rows: self.envdb.t.pkg_deps.insert_all(rows, replace=True)
	    if verbose: print({'changed':len(ch),'same':max(0,len(ex)-len(ch)),'removed': max(0,len(ex)-len(cont_hash))})
    return self.pkgs.insert(dict(name=pkg, version=v(pkg), summary=_get(doc,'summary')), ignore=True)

@patch
def rm_pkg(self:Kosha, pkg:str, ver:str=None):
	'Remove a package and its code snippets from the database.'
	wh = f'package={pkg!r} AND version={ver!r}' if ver else f'package={pkg!r}'
	for t in [self.env_st, self.pkgs]: t.delete(where=wh)
	self.pkg_st.delete_where(where=f"json_extract(metadata,'$.name')={pkg!r}")
	self.env_pd.delete_where(where=f'from_pkg={pkg!r}')
	self.pkgs.delete_where(where=f'name={pkg!r}')

@patch
def update_pkgs(self:Kosha,
    pkgs:str|list=None, # packages to sync; None syncs all installed env packages
    embed=True,         # embed chunks after indexing
    exts=code_exts,     # file extensions to include
    verbose=True,       # print progress
    force=False,        # reindex even if package version already loaded
	parallel=True,		# parallel processing
    **kwargs            # forwarded to update_pkg
):
	'Sync installed packages into the env store; if pkgs is None, syncs all env packages.'
	if verbose: print(f'loading pkgs {pkgs} ...' if pkgs else 'No packages to load.')
	if not pkgs: return
	kw = dict(embed=embed, exts=exts, verbose=verbose, force=force, **kwargs)
	if not parallel:
		for pkg in tqdm(list(set(pkgs)), desc='Updating packages', unit='pkg'): self.update_pkg(pkg, **kw)
	else: arun(parallel_async(self.update_pkg,list(set(pkgs)), pause=0.1, **kw))

In [ ]:
#| export
@patch
@fdelegates(process_content)
def process_repo(self:Kosha, content=None, reembed=False, **kwargs):
	'Embed all documents in the code store, or only those without embeddings if reembed=False.'
	content = content or self.code_st(where=f'embedding is NULL' if not reembed else None)
	return process_content(self.code_st, content, emb_fn=self.emb_doc, **kwargs)

@patch
@fdelegates(dir2files)
def update_repo(self:Kosha,
				dir:Path=None,    # directory to index; defaults to repo root
				embed:bool=True,  # embed chunks after parsing
                exts:str=code_exts,
                verbose=True,       # verbose
                force=False,        # reindex all files regardless of mtime
				**kwargs              # extra args forwarded to dir2files
				):
	'Index or update repo code chunks. Pass files= for incremental update (e.g. from watcher).'
	dir = Path(dir or self.root)
	known = {r['path']: r['uploaded_at'] for r in self.code_st(select='path, uploaded_at') if r['path']}
	all_files = dir2files(str(dir), strict_skip_file_re, strict_skip_folder_re,exts=exts, **kwargs)
	all_str = set(map(str, all_files))
	to_remove = L(known.keys()).filter(lambda k: k not in all_str)
	ch = all_files if force else L(all_files).filter(lambda f: str(f) not in known or known[str(f)] != f.stat().st_mtime)
	if to_remove: self.code_st.delete_where(where=f'path in ({",".join(to_remove.map(repr))})')
	if not ch: return
	if verbose: print(f'syncing files {ch} .....')
	mod_fn = lambda p, n: '.'.join(list(Path(p).relative_to(imp_root(p)).with_suffix('').parts)+([n] if n else []))
	_get = lambda m,k1,k2: m.get(k1,{}).get(k2,'')
	meta_fn = lambda d: d['metadata'] | dict(mod_name=mod_fn(_get(d,'metadata','path'),_get(d,'metadata','name')))
	fn = lambda d: dict(path=_get(d,'metadata','path'),uploaded_at=_get(d,'metadata','uploaded_at'),metadata=meta_fn(d))
	content = L([file_parse(f, assigns=True) for f in tqdm(ch,f'parse files from {dir}')]).concat()
	if not content: return
	content = enrich_chunks(content.map(lambda d: d | fn(d)))
	pl = ','.join(map(repr, L(ch).map(str)))
	ex = L() if force else _content(self.code_st, where=f'path IN ({pl})', f=_slug)
	cont_hash = {_slug(d['content']): d for d in content}
	if dids := set(ex).difference(cont_hash.keys()):
		self.code_st.delete_where(where='id in ("%s")' % '","'.join(dids))
		if verbose: print(f'removed {len(dids)} outdated chunks from code store.')
	to_add = filter_keys(cont_hash, not_(in_(ex)))
	if not to_add: return
	if verbose: print(f'adding {len(to_add)} new/updated chunks to code store...')
	self.process_repo(to_add.values(), embed=embed)
	for f in ch: self.codedb.q(f'update {self.code_st.name} set uploaded_at=? where path=?',[f.stat().st_mtime, str(f)])
	if verbose:print({'changed':len(to_add),'same':max(0,len(ex)-len(to_add)),'removed': max(0,len(ex)-len(cont_hash))})
	own = Path(dir).resolve().name
	rows = [dict(from_module=own, to_pkg=dep, n_files=n) for dep,n in count_imp(ch,own).items() if spec(dep)]
	if rows: self.code_rd.insert_all(rows, replace=True)
	if verbose: print('synced repo')

In [ ]:
#| export
@patch
def prune_old_versions(self:Kosha, pkg:str):
	'Keep only the latest version of a package in the database.'
	if len(vers := self.pkgs(select='version', where=f'name={pkg!r}')) <= 1: return
	latest = sorted(L(vers).attrgot('version'), key=lambda v: tuple(map(int, v.split('.'))))[-1]
	je = lambda k, o='=', v='': f"json_extract(metadata, '$.{k}') {o} {v!r}"
	self.env_st.delete_where(where=f"{je('name')}={pkg!r} AND {je('version')}<>{latest!r}")
	self.pkgs.delete_where(where=f'name={pkg!r} AND version<>{latest!r}')

@patch
def prune_old_pkgs(self:Kosha):
	'Keep only the latest version of each package in the database.'
	L(self.pkgs(select='name')).attrgot('name').map(self.prune_old_versions)

@patch
def pkgs_in_env(self:Kosha, pyproject=False, depth=1) -> list:
	'Intersection of the packages table with packages installed in the environment.'
	st_pkgs, inst_pkgs = L(self.pkgs(select='name, version')), env_pkg_versions(pyproject, depth)
	return st_pkgs.filter(lambda p: p['name'] in inst_pkgs and inst_pkgs[p['name']] == p['version'])

In [ ]:
import kosha.core as _kc, inspect

In [ ]:
_f = Path(inspect.getfile(_kc))
_anchor = _f.parent
while has_init(_anchor): _anchor = _anchor.parent
_mod = '.'.join(_f.relative_to(_anchor).with_suffix('').parts)
assert _mod == 'kosha.core', f'expected kosha.core, got {_mod}'
print('mod_name anchor fix: ok —', _mod)

mod_name anchor fix: ok — kosha.core


In [ ]:
dir2files(repo_root(), strict_skip_file_re, strict_skip_folder_re, exts=code_exts)

[Path('/Users/71293/code/personal/orgs/kosha/kosha/cli.py'), Path('/Users/71293/code/personal/orgs/kosha/kosha/core.py'), Path('/Users/71293/code/personal/orgs/kosha/kosha/graph.py')]

In [ ]:
# k = Kosha()

In [ ]:
pkg='fastcore'
k._is_pkg_ingested(pkg)

In [ ]:
k.env_st(select='content, embedding, json_extract(metadata, "$.mod_name","$.docstring", "$.public_api") as pa',
         where='package="httpx" and json_extract(metadata, "$.public_api")=1', limit=1)

[]

In [ ]:
k.pkgs(order_by='name', limit=1)

[]

In [ ]:
#| eval: false
k.update_pkg('protobuf')

updating pkg: protobuf ...


parse files from protobuf: 100%|██████████| 54/54 [00:00<00:00, 180.00it/s]


{'changed': 765, 'same': 0, 'removed': 0}


{'name': 'protobuf',
 'version': '7.35.1',
 'summary': '',
 'uploaded_at': '2026-07-12 04:11:53'}

In [ ]:
#| eval: false
k.update_pkg('fastlite', force=True)

updating pkg: fastlite ...


parse files from fastlite: 100%|██████████| 4/4 [00:00<00:00, 273.77it/s]

{'changed': 64, 'same': 0, 'removed': 0}


{'name': 'fastlite',
 'version': '0.2.4',
 'summary': 'A bit of extra usability for sqlite',
 'uploaded_at': '2026-07-12 04:11:53'}

In [ ]:
#| eval: false
k.update_repo(skip_file_glob='*.ipynb', force=True)

syncing files [Path('/Users/71293/code/personal/orgs/kosha/kosha/cli.py'), Path('/Users/71293/code/personal/orgs/kosha/kosha/core.py'), Path('/Users/71293/code/personal/orgs/kosha/kosha/graph.py')] .....


parse files from /Users/71293/code/personal/orgs/kosha: 100%|██████████| 3/3 [00:00<00:00, 43.43it/s]

adding 122 new/updated chunks to code store...
{'changed': 122, 'same': 0, 'removed': 0}


synced repo


In [ ]:
k.code_st(select='json_extract(metadata, "$.mod_name","$.docstring", "$.public_api") as pa',
          where='json_extract(metadata, "$.public_api")=1')

[{'pa': '["kosha.cli.sync","Sync repo + env packages + call graph into .kosha/ databases.",true]'},
 {'pa': '["kosha.cli.context","Fan-out semantic search over repo and env, optionally graph-enriched.",true]'},
 {'pa': '["kosha.cli.repo_context","Semantic + keyword search over indexed repo code only.",true]'},
 {'pa': '["kosha.cli.env_context","Semantic search over indexed env packages only.",true]'},
 {'pa': '["kosha.cli.ni","Node info: callers, callees, co_dispatched, pagerank for a single graph node.",true]'},
 {'pa': '["kosha.cli.watch","Live incremental re-index on file changes. Blocking — Ctrl-C to stop.",true]'},
 {'pa': '["kosha.cli.public_api","List public API entries for a package (respects __all__ + @patch methods).",true]'},
 {'pa': '["kosha.cli.api_paths","Shortest call-graph paths from from_pkg public API to to_pkg public API.",true]'},
 {'pa': '["kosha.cli.dep_stack","BFS dependency layers from seed packages, ordered by coupling strength.",true]'},
 {'pa': '["kosha.cli.t

In [ ]:
k.code_st(select='content',where='json_extract(metadata, "$.mod_name")="kosha.core.Kosha.__init__"')

[{'content': "def __init__(self, dir: Path = None, install_skill: bool = False, xdg_dir: Path = None, efn=static_embedder):\n\t\tself.root, self.xdg = Path(dir or repo_root() or '.'), xdg_dir or xdg_data_home()\n\t\tself.efn = bind(efn, md=self.xdg/'model')\n\t\tself.emb_doc, self.emb_query = emb_doc(self.efn), emb_query(self.efn)\n\t\tif install_skill: mv_skill_md(dir=self.root, dry_run=False)\n\t\tself.cp, self.ce = self.root.joinpath('.kosha','code.db'), self.xdg.joinpath('kosha','env.db')\n\t\tfor p in (self.cp, self.ce): p.parent.mkdir(parents=True, exist_ok=True)\n\t\tself.codedb, self.envdb = database(self.cp), database(self.ce)\n\t\tself.code_st,self.env_st = self.codedb.get_store(path=str,hash=True),self.envdb.get_store(package=str,hash=True)\n\t\tself.pkg_st, self.pkgs = self.envdb.get_store('pkg_store', hash=True), self.envdb.t.packages\n\t\tself.env_pd, self.code_rd = self.envdb.t.pkg_deps, self.codedb.t.repo_deps\n\t\tself.pkgs.create(name=str, version=str, summary=str, up

In [ ]:
#| eval: false
k.update_pkgs(set(env_pkg_versions()), embed=False, force=True)

loading pkgs {'fastprogress', 'fastcore', 'fastaudit', 'anyio', 'codesigs', 'yake', 'litesearch', 'chonkie', 'pdf_oxide', 'watchfiles', 'notebook', 'python-fasthtml', 'pyskills', 'pandas', 'Jinja2', 'pillow', 'usearch', 'model2vec', 'onnxruntime', 'tokenizers', 'fastlite', 'pyan3'} ...


Updating packages:   0%|          | 0/22 [00:00<?, ?pkg/s]

updating pkg: fastprogress ...



parse files from fastprogress: 100%|██████████| 6/6 [00:00<00:00, 289.90it/s]


{'changed': 59, 'same': 0, 'removed': 0}
updating pkg: fastcore ...


...

{'changed': 86, 'same': 0, 'removed': 0}
updating pkg: fastlite ...
package {'name': 'fastlite', 'version': '0.2.4'} already loaded.
forcing an update to pkg {'name': 'fastlite', 'version': '0.2.4'}



Updating packages:  95%|█████████▌| 21/22 [00:55<00:01,  1.93s/pkg]

{'changed': 64, 'same': 0, 'removed': 0}
updating pkg: pyan3 ...



Updating packages: 100%|██████████| 22/22 [00:55<00:00,  2.53s/pkg]

{'changed': 241, 'same': 0, 'removed': 0}


In [ ]:
#| eval: false
k2.update_pkgs(set(env_pkg_versions()))

loading pkgs {'fastprogress', 'fastcore', 'fastaudit', 'anyio', 'codesigs', 'yake', 'litesearch', 'chonkie', 'pdf_oxide', 'watchfiles', 'notebook', 'python-fasthtml', 'pyskills', 'pandas', 'Jinja2', 'pillow', 'usearch', 'model2vec', 'onnxruntime', 'tokenizers', 'fastlite', 'pyan3'} ...
updating pkg: fastprogress ...
package {'name': 'fastprogress', 'version': '1.1.6'} already loaded.
updating pkg: fastcore ...
package {'name': 'fastcore', 'version': '2.0.1'} already loaded.
updating pkg: fastaudit ...
package {'name': 'fastaudit', 'version': '0.2.5'} already loaded.
updating pkg: anyio ...
package {'name': 'anyio', 'version': '4.14.1'} already loaded.
updating pkg: codesigs ...
package {'name': 'codesigs', 'version': '0.0.3'} already loaded.
updating pkg: yake ...
package {'name': 'yake', 'version': '0.7.3'} already loaded.
updating pkg: litesearch ...
package {'name': 'litesearch', 'version': '0.0.30'} already loaded.
updating pkg: chonkie ...
package {'name': 'chonkie', 'version': '1

In [ ]:
k.env_st(where='embedding is NULL', select='count(*)')

[{'count(*)': 0}]

In [ ]:
k2.env_st(where='embedding is NULL', select='count(*)')

[{'count(*)': 0}]

In [ ]:
#| eval: false
k.process_env()

In [ ]:
#| eval: false
k.update_pkgs(['fastcore','litesearch','fastlite','apsw','ghapi','httpx'], embed=True)

In [ ]:
#| eval: false
k2.update_pkgs(['fastcore','litesearch','fastlite','apsw','ghapi','httpx'], embed=True)

loading pkgs ['fastcore', 'litesearch', 'fastlite', 'apsw', 'ghapi', 'httpx'] ...


Updating packages:   0%|          | 0/6 [00:00<?, ?pkg/s]

updating pkg: apsw ...



Updating packages:  17%|█▋        | 1/6 [00:02<00:12,  2.52s/pkg]

{'changed': 437, 'same': 0, 'removed': 0}
updating pkg: httpx ...



Updating packages:  33%|███▎      | 2/6 [00:04<00:08,  2.24s/pkg]

{'changed': 514, 'same': 0, 'removed': 0}
updating pkg: fastcore ...
package {'name': 'fastcore', 'version': '2.0.1'} already loaded.
updating pkg: ghapi ...



Updating packages: 100%|██████████| 6/6 [00:05<00:00,  1.09pkg/s]

{'changed': 136, 'same': 0, 'removed': 0}
updating pkg: litesearch ...
package {'name': 'litesearch', 'version': '0.0.30'} already loaded.
updating pkg: fastlite ...
package {'name': 'fastlite', 'version': '0.2.4'} already loaded.


## Structured Search
> `parseq` strips `key:value` filter tokens from a query string. `filt2wh` maps them to SQL WHERE clauses. `Kosha.context` fans out searches across detected packages and merges results with chained RRF.

In [ ]:
#| export
_filter_keys = frozenset({'file', 'files', 'path', 'paths', 'package', 'packages', 'lang', 'langs', 'type', 'types'})
_filter_pat = re.compile(r'\b(' + '|'.join(_filter_keys) + r'):(\S+)')
_filter_norm = {'files': 'file', 'paths': 'path', 'packages': 'package', 'langs': 'lang', 'types': 'type'}

def parseq(q: str) -> tuple:
    'Parse key:value filter tokens from a query. Returns (bare_query, filters_dict).'
    filters = defaultdict(list)
    for m in _filter_pat.finditer(q):
        key = _filter_norm.get(m.group(1), m.group(1))
        filters[key].extend(m.group(2).split(','))
    return _filter_pat.sub('', q).strip() or q, filters

In [ ]:
parseq('how do I merge dicts package:fastcore path:data*')

('how do I merge dicts',
 defaultdict(list, {'package': ['fastcore'], 'path': ['data*']}))

In [ ]:
assert parseq('stripe payments package:fasthtml file:routes*') == ('stripe payments', {'package': ['fasthtml'], 'file': ['routes*']})
assert parseq('package:monsterui package:fasthtml create a table') == ('create a table', {'package': ['monsterui', 'fasthtml']})
assert parseq('no filters here') == ('no filters here', {})
assert parseq('type:FunctionDef lang:py') == ('type:FunctionDef lang:py', {'type': ['FunctionDef'], 'lang': ['py']})
assert parseq('package:only')[0] == 'package:only'
# plural forms + comma-separated values
assert parseq('merge dicts packages:fastcore,litesearch') == ('merge dicts', {'package': ['fastcore', 'litesearch']})
assert parseq('search paths:basics,core,data') == ('search', {'path': ['basics', 'core', 'data']})
assert parseq('query packages:fastcore,litesearch paths:basics,core') == ('query', {'package': ['fastcore', 'litesearch'], 'path': ['basics', 'core']})
assert parseq('langs:py,js files:routes*,api*') == ('langs:py,js files:routes*,api*', {'lang': ['py', 'js'], 'file': ['routes*', 'api*']})
print('parse_query: all assertions pass')

parse_query: all assertions pass


In [ ]:
from litesearch import pre, clean

In [ ]:
# Regression: underscore in bare query must not survive into FTS token
assert clean('with_uv') == 'with uv', f'clean() did not strip underscore: {clean("with_uv")!r}'
result = pre('with_uv', wc=False, wide=False, extract_kw=False)
assert result is not None, 'pre() returned None for non-empty query'
assert '_' not in result, f'underscore survived pre(): {result!r}'
print('pre() underscore sanitisation: pass')

pre() underscore sanitisation: pass


In [ ]:
#| export
_glob2like = {'*': '%', '?': '_'}
def _glob_to_like(pat: str) -> str:
    'Convert a glob pattern (*, ?) to a SQL LIKE pattern with surrounding % anchors.'
    p = pat.replace('*', '%').replace('?', '_')
    if not p.startswith('%'): p = '%' + p
    if not p.endswith('%'): p += '%'
    return p

_ext = {'py': '.py', 'js': '.js', 'ts': '.ts', 'jsx': '.jsx', 'tsx': '.tsx', 'rb':'.rb','rs':'.rs','go':'.go',
        'java':'.java','cs':'.cs', 'swift':'.swift','kt':'.kt','lua':'.lua','php':'.php','python':'.py',
        'javascript':'.js','typescript':'.ts','ruby':'.rb','rust':'.rs','golang':'.go'}

def filt2wh(filters: dict, store: str = 'code') -> str | None:
	'Convert a filter dict (from parse_query) to a SQL WHERE clause for the given store (code or env).'
	_get, je = lambda k: filters.get(k,[]), lambda k, o='=', v='': f"json_extract(metadata, '$.{k}') {o} {v!r}"
	list_or = lambda it: ['(%s)'%' OR '.join(ls)] if (ls:=listify(it)) else []
	c, files, paths, langs = [], _get('file'), _get('path'), _get('lang')
	path_wh, lang_chk = list(map(_glob_to_like, files + paths)), list(map(lambda l: _ext.get(l, f'.{l}'), langs))
	lang_wh = list_or(map(lambda lc: je('lang',v=lc), lang_chk))
	if store == 'code': c = list_or(map(lambda p: f'path LIKE {p!r}',path_wh)) + lang_wh
	elif store == 'env':
		c = list_or(map(lambda p: je('path','like',p),path_wh)) + lang_wh
		if pkgs:=_get('package'): c += list_or(map(lambda p: f'package={p!r}',pkgs))
	c += list_or(map(lambda r: je('type',v=r), _get('type')))
	return ' AND '.join(c) if c else None

In [ ]:
filt2wh(parseq('how do I merge dicts package:fastcore path:data*')[1], 'env')

"(json_extract(metadata, '$.path') like '%data%') AND (package='fastcore')"

In [ ]:
filt2wh({'type': ['FunctionDef']}, 'code')

"(json_extract(metadata, '$.type') = 'FunctionDef')"

In [ ]:
assert filt2wh({'file': ['routes*']}, 'code') == "(path LIKE '%routes%')"
assert filt2wh({'path': ['api/*']}, 'code') == "(path LIKE '%api/%')"
assert filt2wh({'lang': ['py']}, 'code') == "(json_extract(metadata, '$.lang') = '.py')"
assert filt2wh({'lang': ['python']}, 'code') == "(json_extract(metadata, '$.lang') = '.py')"
assert filt2wh({'package': ['fasthtml']}, 'env') == "(package='fasthtml')"
assert filt2wh({'package': ['fasthtml', 'monsterui']}, 'env') == "(package='fasthtml' OR package='monsterui')"
assert filt2wh({'type': ['FunctionDef']}, 'code') == "(json_extract(metadata, '$.type') = 'FunctionDef')"
assert filt2wh({}, 'code') is None
assert filt2wh({}, 'env') is None
assert filt2wh({'package': ['fasthtml']}, 'code') is None
print('filters_to_where: all assertions pass')

filters_to_where: all assertions pass


In [ ]:
#| export
@patch
def env_context(self:Kosha,
				q:str,               	# query to search (supports key:value filters)
				emb_q:str=None,     	# query to embed; defaults to bare query after filter parsing
				wide:bool=False,    	# whether to use wide search
                columns:str='content,metadata,package',			# comma separated columns string to return from search
                where:str=None,			# additional where clause to filter search results
				sys_wide=True,
				**kw					# additional args to pass to db.search
				) -> L:
	'Code search through the database to find relevant code snippets.'
	raw, fs = parseq(q)
	fq, emb = pre(raw, wide=wide, extract_kw=False), self.emb_query(emb_q or raw)
	all_p = set(raw.split()).intersection(self.pkgs2consider(sys_wide)) if 'package' not in fs else []
	for p in all_p: fs['package'].append(p)
	wh = ' AND '.join(map(lambda p: f'({p})', L(filt2wh(fs, 'env'), where).filter(true)))
	fn = lambda r: r | dict(metadata=jl(r['metadata'])) if 'metadata' in r else r
	return L(self.envdb.search(fq, emb.tobytes(), columns.split(','), where=wh, **kw)).map(fn)

@patch
def pkgs2consider(self: Kosha, sys_wide=True) -> set:
	'Determine which packages to consider for search intersecting packages table and the environment.'
	ex_pkgs = {d['name']: d['version'] for d in self.pkgs(select='name, version')}
	if sys_wide: return ex_pkgs.keys()
	env_pkgs = env_pkg_versions()
	return {p for p in env_pkgs if p in ex_pkgs and env_pkgs[p] == ex_pkgs[p]}

In [ ]:
#| export
@patch
def repo_context(self:Kosha,
                q:str,                          # query to search (supports key:value filters)
                emb_q:str=None,                 # query to embed; defaults to bare query after filter parsing
                wide:bool=False,                # use wide (OR) FTS search
                columns:str='content,path,metadata', # columns to return
                where:str=None,                 # extra SQL filter
                **kw                            # additional args to pass to db.search
) -> L:
	'Semantic + keyword search through indexed repo code.'
	raw, fs = parseq(q)
	fq, emb = pre(raw, wide=wide, extract_kw=False), self.emb_query(emb_q or raw)
	wh = filt2wh(fs, 'code')
	if where and wh: wh = f'({where}) AND ({wh})'
	elif where: wh = where
	fn = lambda r: r | dict(metadata=jl(r['metadata'])) if 'metadata' in r else r
	return L(self.codedb.search(fq, emb.tobytes(), columns.split(','), where=wh, **kw)).map(fn)

@patch
async def awatch_repo(self:Kosha, dir:Path=None, **kw):
	'Async watcher: re-indexes changed files on every watchfiles event.'
	from watchfiles import awatch
	dir = Path(dir or self.root)
	async for changes in awatch(str(dir)): self.update_repo(dir, files=L({c[1] for c in changes}), **kw)

@patch
def watch_repo(self:Kosha, dir:Path=None, **kw):
	'Block and watch repo for changes, re-indexing incrementally. Ctrl-C to stop.'
	arun(self.awatch_repo(dir, **kw))

In [ ]:
#| export
@patch
def pkg_context(self:Kosha,
                q:str, # the search query
                limit:int=10, # limits
) -> L:
	'FTS5+vector search over package descriptions in pkg_store.'
	fq, emb = pre(q, extract_kw=False), self.emb_query(q).tobytes()
	fn = lambda r: r | dict(metadata=jl(r['metadata'])) if isinstance(r.get('metadata'), str) else r
	return L(self.envdb.search(fq, emb, columns=['content','metadata'], table_name='pkg_store', limit=limit)).map(fn)

In [ ]:
first(k.repo_context('how do I parse python ast', limit=1))

{'rowid': 18,
 'content': 'def parse(code=None, p=None):\n    "Parse source, tag parents, return (tree, imp, top_fns, all_fns, imp2mod). Cached — called freely."\n    if not code and not p: return None, {}, set(), set(), {}\n    try: tree = ast.parse(Path(p).read_text(errors=\'replace\') if not code else code)\n    except SyntaxError: return None, {}, set(), set(), {}\n    [setattr(c,\'parent\',n) for n in ast.walk(tree) for c in ast.iter_child_nodes(n)]\n    is_top = lambda n: isinstance(getattr(n,\'parent\',None), ast.Module)\n    is_fn  = lambda n: isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef))\n    is_nm  = lambda n: isinstance(n, ast.Name)\n    imp, imp2mod, top = {}, {}, set()\n    # Single pass: imports + top-level fn names\n    for n in ast.walk(tree):\n        if is_fn(n) and is_top(n): top.add(n.name)\n        elif isinstance(n, ast.Import):\n            for a in n.names: imp[a.asname or a.name.split(\'.\')[0]] = a.name.split(\'.\')[0]\n        elif isinsta

In [ ]:
k.env_context('payment page with fasthtml', limit=10).map(lambda d: d['metadata']['path'])

['/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fasthtml/stripe_otp.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fasthtml/stripe_otp.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/notebook/static/notebook_core.2ef41b6716a70667.js', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fasthtml/stripe_otp.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/notebook/static/notebook_core.2ef41b6716a70667.js', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/notebook/static/notebook_core.2ef41b6716a70667.js', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/notebook/static/2613.218b5a2f93bbe3fa.js', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fasthtml/stripe_otp.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/notebook/static/8924.a6f1905e51b71

In [ ]:
k2.env_context('payment page with fasthtml', limit=10).map(lambda d: d['metadata']['path'])

['/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fasthtml/stripe_otp.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fasthtml/stripe_otp.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fasthtml/jupyter.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xml.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fasthtml/fastapp.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastprogress/core.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fasthtml/jupyter.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/notebook/static/notebook_core.2ef41b6716a70667.js', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/notebook/static/953.67fec00f0e0dd7de.js', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fasthtml

In [ ]:
k.env_context('payment page with fasthtml package:fastcore', limit=10).map(lambda d: d['metadata']['path'])

['/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xml.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xml.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xml.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xml.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/nbio.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/net.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xml.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/net.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/net.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/net.py']

In [ ]:
k2.env_context('payment page with fasthtml package:fastcore', limit=10).map(lambda d: d['metadata']['path'])

['/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xml.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/nbio.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/nbio.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xml.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/imghdr.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xml.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/nbio.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xtras.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/imghdr.py', '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/nbio.py']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()